# PINN Review — Part I: Papers 6–10 (2019–2020)

Executable reproductions for PPINN, VPINN, high-speed flows, Bayesian PINNs, and conservative PINNs.

Source: Physics-Informed Neural Networks 2019–2026 review article.


## Shared Setup


In [ ]:
import math, time, random, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)
SEED=7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cpu'

def rel_l2(yhat,y):
    return float(torch.linalg.norm(yhat-y)/torch.linalg.norm(y))

class MLP(nn.Module):
    def __init__(self, din=1, dout=1, width=32, depth=3, dropout=0.0, act=nn.Tanh):
        super().__init__(); layers=[]; d=din
        for _ in range(depth):
            layers += [nn.Linear(d,width), act()]
            if dropout: layers += [nn.Dropout(dropout)]
            d=width
        layers += [nn.Linear(d,dout)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def grad(y,x,order=1):
    g=y
    for _ in range(order):
        g=torch.autograd.grad(g,x,torch.ones_like(g),create_graph=True,retain_graph=True)[0]
    return g

def train(loss_fn, model, steps=600, lr=2e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); hist=[]
    t=time.time()
    for i in range(steps):
        opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
        if i % max(1,steps//100)==0: hist.append(float(loss.detach()))
    return hist, time.time()-t

results=[]
def record(paper,method,error,seconds,notes):
    results.append(dict(paper=paper,method=method,relative_L2=error,seconds=seconds,notes=notes))


## Paper 6 — Meng et al. (2019/2020): PPINN — Parareal Time Decomposition


In [ ]:
def rhs(t,u): return 1+(math.pi/2)*math.cos(math.pi*t/2)
def G(u,t,dt): return u+dt*rhs(t,u)
def F(u,t,dt,n=20):
    h=dt/n; v=u
    for j in range(n):
        tj=t+j*h; k1=rhs(tj,v); k2=rhs(tj+h/2,v+h*k1/2); k3=rhs(tj+h/2,v+h*k2/2); k4=rhs(tj+h,v+h*k3); v+=h*(k1+2*k2+2*k3+k4)/6
    return v
N=10; dt=1.; U=np.zeros(N+1)
for i in range(N): U[i+1]=G(U[i],i,dt)
errs=[]; st=time.time()
for k in range(5):
    old=U.copy(); new=np.zeros_like(U)
    for i in range(N): new[i+1]=G(new[i],i,dt)+F(old[i],i,dt)-G(old[i],i,dt)
    U=new; exact=np.arange(N+1)+np.sin(math.pi*np.arange(N+1)/2); errs.append(np.linalg.norm(U-exact)/np.linalg.norm(exact))
t=time.time()-st; record('Meng et al. PPINN','parareal mechanism',errs[-1],t,'coarse Euler/fine RK4')
plt.figure(figsize=(7,4)); plt.semilogy(range(1,6),errs,'o-'); plt.xlabel('parareal iteration'); plt.ylabel('relative error'); plt.title('PPINN prediction-correction mechanism'); plt.show()


## Paper 7 — Kharazmi et al. (2019): VPINN — Variational Residuals


In [ ]:
torch.manual_seed(SEED)
# strong result reuse as base; weak-form model
v=MLP(width=28,depth=3); q=torch.linspace(0,1,100).reshape(-1,1).requires_grad_(True); b=torch.tensor([[0.],[1.]])
def weak_loss():
    u=v(q); ux=grad(u,q); vals=[]
    for k in range(1,7):
        phi=torch.sin(k*math.pi*q); phix=k*math.pi*torch.cos(k*math.pi*q); f=-math.pi**2*torch.sin(math.pi*q)
        # -int u' phi' = int f phi
        vals.append((torch.trapz((-ux*phix-f*phi).squeeze(),q.squeeze()))**2)
    return torch.stack(vals).mean()+50*(v(b)**2).mean()
h,t=train(weak_loss,v,300)
with torch.no_grad(): yv=v(xt)
ev=rel_l2(yv,torch.sin(math.pi*xt)); record('Kharazmi et al. VPINN','VPINN',ev,t,'six sine test functions')
plt.figure(figsize=(7,4)); plt.plot(xt,torch.sin(math.pi*xt),label='exact'); plt.plot(xt,yv,'--',label='VPINN'); plt.legend(); plt.title(f'Variational PINN, relative L2={ev:.2e}'); plt.show()


## Paper 8 — Mao et al. (2020): High-Speed Flows with PINNs


In [ ]:
torch.manual_seed(SEED)
def rho(z):
    x,t=z[:,0:1],z[:,1:2]; return torch.where(x<.5+.1*t,torch.tensor(1.4),torch.tensor(1.0))
def fit_contact(cluster=False):
    z=torch.rand(500,2)
    if cluster:
        tt=torch.rand(250,1); xx=.5+.1*tt+.03*torch.randn_like(tt); z=torch.cat([z[:250],torch.cat([xx.clamp(0,1),tt],1)],0)
    y=rho(z); m=MLP(2,1,30,3); h,t=train(lambda:((m(z)-y)**2).mean(),m,300); return m,t
ma,ta=fit_contact(False); mc,tc=fit_contact(True)
ztest=torch.rand(3000,2); yy=rho(ztest)
with torch.no_grad(): ea=rel_l2(ma(ztest),yy); ec=rel_l2(mc(ztest),yy)
record('Mao et al. 2020','random points proxy',ea,ta,'contact discontinuity'); record('Mao et al. 2020','clustered points proxy',ec,tc,'contact discontinuity')
line=torch.linspace(0,1,300).reshape(-1,1); zline=torch.cat([line,torch.full_like(line,.8)],1)
plt.figure(figsize=(7,4)); plt.plot(line,rho(zline),label='exact'); plt.plot(line,ma(zline).detach(),'--',label='random'); plt.plot(line,mc(zline).detach(),':',label='clustered'); plt.legend(); plt.title('Sampling near a discontinuity'); plt.show()


## Paper 9 — Yang et al. (2020): B-PINN — Bayesian Physics-Informed Neural Networks


In [ ]:
torch.manual_seed(SEED)
td=torch.linspace(0,1,25).reshape(-1,1); yd=torch.exp(-1.7*td)+.08*torch.randn_like(td)
lam=torch.tensor(1.0,requires_grad=True); opt=torch.optim.Adam([lam],.05); st=time.time()
for _ in range(250):
    loss=((torch.exp(-lam*td)-yd)**2).mean()+.001*lam**2; opt.zero_grad(); loss.backward(); opt.step()
# curvature of negative log posterior
L=((torch.exp(-lam*td)-yd)**2).sum()/(2*.08**2)+lam**2/2
H=torch.autograd.grad(torch.autograd.grad(L,lam,create_graph=True)[0],lam)[0].detach(); sd=float(torch.sqrt(1/H)); mean=float(lam.detach()); t=time.time()-st
pred=torch.exp(-mean*xt); e=rel_l2(pred,torch.exp(-1.7*xt)); record('Yang et al. B-PINN','Laplace Bayesian inverse',e,t,f'lambda={mean:.3f}±{sd:.3f}')
plt.figure(figsize=(7,4)); plt.scatter(td,yd,label='noisy data'); plt.plot(xt,torch.exp(-1.7*xt),'k--',label='truth'); plt.plot(xt,pred,label='posterior mean'); plt.fill_between(xt.squeeze(),torch.exp(-(mean+2*sd)*xt).squeeze(),torch.exp(-(mean-2*sd)*xt).squeeze(),alpha=.2,label='parameter uncertainty'); plt.legend(); plt.title('Bayesian inverse parameter proxy'); plt.show()


## Paper 10 — Jagtap et al. (2020): cPINN — Conservative Domain Decomposition


In [ ]:
torch.manual_seed(SEED)
cl=MLP(width=20,depth=2); cr=MLP(width=20,depth=2); xl=torch.linspace(0,.5,35).reshape(-1,1).requires_grad_(True); xr=torch.linspace(.5,1,35).reshape(-1,1).requires_grad_(True)
pars=list(cl.parameters())+list(cr.parameters()); opt=torch.optim.Adam(pars,2e-3); st=time.time()
for _ in range(300):
    ul,ur=cl(xl),cr(xr); rl=grad(ul,xl,2)+math.pi**2*torch.sin(math.pi*xl); rr=grad(ur,xr,2)+math.pi**2*torch.sin(math.pi*xr)
    xi=torch.tensor([[.5]],requires_grad=True); uli,uri=cl(xi),cr(xi); flux=(grad(uli,xi)-grad(uri,xi))**2
    loss=(rl**2).mean()+(rr**2).mean()+50*(cl(torch.tensor([[0.]]))**2+cr(torch.tensor([[1.]]))**2).mean()+20*((uli-uri)**2+flux).mean()
    opt.zero_grad(); loss.backward(); opt.step()
t=time.time()-st
with torch.no_grad(): yc=torch.where(xt<=.5,cl(xt),cr(xt))
e=rel_l2(yc,torch.sin(math.pi*xt)); record('Jagtap et al. cPINN','cPINN',e,t,'solution + flux interface')
plt.figure(figsize=(7,4)); plt.plot(xt,torch.sin(math.pi*xt),label='exact'); plt.plot(xt,yc,'--',label='cPINN'); plt.axvline(.5,color='gray',ls=':'); plt.legend(); plt.title('Conservative interface stitching'); plt.show()
